# Checking Data Quality in a Synthetic Histology Dataset
**Amber Locasto | IAN 630 | Track B: Pathway systems project**

## 1. Question and purpose
Can simple checks find missing information, inconsistent report values, and workflow timing errors in a synthetic histology dataset?

A histology lab handles tissue specimens and produces reports. Incomplete or inconsistent records can make tracking specimens and measuring turnaround time harder. This project shows which records a lab staff member should review. It does not diagnose disease or verify the physical identity of tissue.

The scope is a small descriptive analysis and a proposed review workflow. There is no prediction model or automatic correction.

## 2. Data and background
The instructor supplied synthetic Barrett’s esophagus surveillance data. The README describes one row as one specimen/report record. A patient can have several reports.

- `data/barrett_surveillance.csv`: the report data used for the checks.
- `data/barrett_surveillance_dq_flags.csv`: the instructor's catalog of injected issues, used only for evaluation.
- `histology_systems_brief.md`: the separate Track B design document.

All report records are included. Only intervals with usable endpoints are included in duration summaries. These are fictional records, not a real patient extract or tissue images.

The data families are clinical report fields, laboratory workflow information, and synthetic secondary/research data. Examples include dysplasia category, tissue type, and processing timestamps. The CSV is not a FHIR dataset or an actual laboratory information system (LIS) feed.

The relevant standards here are the documented field names, category labels, units, and ISO-style timestamps. A real implementation would also need agreed terminology and an interface specification. This project does not implement a clinical messaging standard.

## 3. Load and explore
I use pandas to read the file and keep an unchanged copy. Identifiers are read as text. The table below checks the size of the dataset and basic identifier quality.

In [1]:
import pandas as pd

# Run this notebook from the histology_project folder.
df_raw = pd.read_csv(
    "data/barrett_surveillance.csv",
    dtype={"patient_id": "string", "report_id": "string"}
)
df = df_raw.copy()

identifier_summary = pd.Series({
    "Report records": len(df),
    "Unique patients": df["patient_id"].nunique(),
    "Unique report IDs": df["report_id"].nunique(),
    "Missing patient IDs": df["patient_id"].isna().sum(),
    "Missing report IDs": df["report_id"].isna().sum(),
    "Records sharing a report ID": (
        df["report_id"].notna()
        & df["report_id"].duplicated(keep=False)
    ).sum()
})
print(identifier_summary)

Report records                 158
Unique patients                 80
Unique report IDs              158
Missing patient IDs              0
Missing report IDs               0
Records sharing a report ID      0
dtype: int64


### Missing values
The percentage is based on all report records. A blank is recorded as missing; this alone does not tell us whether a field was required for that report.

In [2]:
missing_summary = pd.DataFrame({
    "Missing records": df.isna().sum(),
    "Percent of all records": (df.isna().mean() * 100).round(2)
})
print(missing_summary[missing_summary["Missing records"] > 0])

                            Missing records  Percent of all records
processed_at                              8                    5.06
prague_c_cm                               6                    3.80
second_pathologist_confirm                8                    5.06


### Category labels
A frequency table makes it easy to see the exact labels. These counts describe reports, not unique patients.

In [3]:
for column in ["tissue_type", "procedure", "dysplasia_grade"]:
    print(column)
    print(df[column].value_counts(dropna=False).rename("Report count"))

tissue_type
tissue_type
barrett_mucosa                113
ge_junction                    24
distal_esophageal_squamous     13
gastric_cardia                  8
Name: Report count, dtype: int64
procedure
procedure
forceps_biopsy    152
emr                 6
Name: Report count, dtype: int64
dysplasia_grade
dysplasia_grade
none          84
indefinite    41
low_grade     27
high_grade     4
LGD            2
Name: Report count, dtype: int64


## 4. Methods: seven simple checks
The checks use the dataset documentation. They flag records for review and do not change original values.

| Check | Rule |
|---|---|
| Missing processing time | `processed_at` is blank |
| Missing Prague C | `prague_c_cm` is blank |
| Unexpected dysplasia label | Label is outside the four expected categories |
| Prague consistency | Circumferential length C exceeds maximum length M |
| Processing order | Processing completion occurs after slide readiness |
| Missing required confirmation | Confirmation is blank for a recognized dysplasia category other than none |
| Confirmation inconsistency | Confirmation is True while dysplasia is none |

C and M are the circumferential and maximum lengths recorded in the report. These checks assess their relationship, not whether the measurements are clinically correct.

The confirmation rules are teaching-dataset rules, not universal clinical requirements. Unexpected dysplasia labels are flagged separately rather than guessed or silently corrected.

### Prepare timestamps
Python needs datetime values to compare times. Unreadable values become `NaT`. I distinguish them from existing blanks. Original text stays in `df_raw`.

In [4]:
time_columns = [
    "collected_at", "received_at", "processed_at",
    "slide_ready_at", "reported_at"
]

for column in time_columns:
    df[column] = pd.to_datetime(df_raw[column], errors="coerce")

unreadable_times = (
    df_raw[time_columns].notna() & df[time_columns].isna()
).sum()
print(unreadable_times.rename("Unreadable timestamps"))

collected_at      0
received_at       0
processed_at      0
slide_ready_at    0
reported_at       0
Name: Unreadable timestamps, dtype: int64


### Apply the checks
Each column in `flags` is one rule. True means the record needs review. For processing order, both timestamps must be available. A missing endpoint is not evidence that the order was correct.

In [5]:
expected_grades = ["none", "indefinite", "low_grade", "high_grade"]

# Convert confirmation values to text only for comparison.
confirmation = (
    df["second_pathologist_confirm"].astype("string").str.lower()
)
requires_confirmation = df["dysplasia_grade"].isin(
    ["indefinite", "low_grade", "high_grade"]
)

flags = pd.DataFrame(index=df.index)
flags["Missing processing time"] = df_raw["processed_at"].isna()
flags["Missing Prague C"] = df["prague_c_cm"].isna()
flags["Unexpected dysplasia label"] = (
    df["dysplasia_grade"].notna()
    & ~df["dysplasia_grade"].isin(expected_grades)
)
flags["Prague C exceeds M"] = (
    df["prague_c_cm"].notna()
    & df["prague_m_cm"].notna()
    & (df["prague_c_cm"] > df["prague_m_cm"])
)
flags["Processing after slides ready"] = (
    df["processed_at"].notna()
    & df["slide_ready_at"].notna()
    & (df["processed_at"] > df["slide_ready_at"])
)
flags["Required confirmation missing"] = (
    requires_confirmation & confirmation.isna()
)
flags["Confirmation True with no dysplasia"] = (
    df["dysplasia_grade"].eq("none")
    & confirmation.eq("true").fillna(False)
)

print(flags.sum().rename("Flagged records").to_frame())
print("Records with at least one flag:", flags.any(axis=1).sum())

                                     Flagged records
Missing processing time                            8
Missing Prague C                                   6
Unexpected dysplasia label                         2
Prague C exceeds M                                 3
Processing after slides ready                      3
Required confirmation missing                      7
Confirmation True with no dysplasia                2
Records with at least one flag: 30


### Review individual records
This table links each flag to a report. It shows one row per report and check, so a report with multiple flags appears more than once. It is a review list, not a list of confirmed corrections.

In [6]:
review_rows = []
for check in flags.columns:
    rows = df.loc[flags[check], ["patient_id", "report_id"]].copy()
    rows["Reason for review"] = check
    review_rows.append(rows)

review_table = pd.concat(review_rows, ignore_index=True)
print(review_table.sort_values("report_id"))

   patient_id     report_id                    Reason for review
22   SYN-0001  SYN-0001-R03        Required confirmation missing
0    SYN-0003  SYN-0003-R01              Missing processing time
16   SYN-0003  SYN-0003-R01                   Prague C exceeds M
8    SYN-0004  SYN-0004-R01                     Missing Prague C
17   SYN-0012  SYN-0012-R01                   Prague C exceeds M
1    SYN-0013  SYN-0013-R04              Missing processing time
23   SYN-0015  SYN-0015-R02        Required confirmation missing
2    SYN-0016  SYN-0016-R01              Missing processing time
19   SYN-0019  SYN-0019-R02        Processing after slides ready
24   SYN-0020  SYN-0020-R02        Required confirmation missing
3    SYN-0022  SYN-0022-R01              Missing processing time
4    SYN-0029  SYN-0029-R02              Missing processing time
9    SYN-0032  SYN-0032-R02                     Missing Prague C
10   SYN-0033  SYN-0033-R01                     Missing Prague C
5    SYN-0036  SYN-0036-R

### Describe receipt-to-report time
For a basic turnaround summary, I use one interval: laboratory receipt to final report. Both endpoints must be present and the report must not precede receipt. This does not require every intermediate timestamp to be valid.

Elapsed time includes waiting as well as work. No performance target is applied.

In [7]:
usable = (
    df["received_at"].notna()
    & df["reported_at"].notna()
    & (df["reported_at"] >= df["received_at"])
)
report_hours = (
    (df["reported_at"] - df["received_at"]).dt.total_seconds() / 3600
)
report_hours = report_hours[usable]

print(pd.Series({
    "Usable records": len(report_hours),
    "Excluded records": len(df) - len(report_hours),
    "Median hours": report_hours.median(),
    "Minimum hours": report_hours.min(),
    "Maximum hours": report_hours.max()
}).round(2))

Usable records      158.00
Excluded records      0.00
Median hours         46.42
Minimum hours        15.12
Maximum hours        94.60
dtype: float64


## 5. Evaluate against the answer key
The answer key is loaded after the checks. I compare the report ID, affected field, and issue type. Prague swaps affect both C and M, so the comparison counts affected reports once using the C entry.

One loop replaces the repeated evaluation cells. For each check it counts documented issues found, documented issues missed, and flags without a matching entry. The seven checks cover selected issues; they do not cover every injected error.

In [8]:
answer_key = pd.read_csv(
    "data/barrett_surveillance_dq_flags.csv", dtype="string"
)

# Each entry identifies the answer-key field and issue type for one check.
key_rules = [
    ("Missing processing time", "processed_at", "missing"),
    ("Missing Prague C", "prague_c_cm", "missing"),
    ("Unexpected dysplasia label", "dysplasia_grade", "unrealistic"),
    ("Prague C exceeds M", "prague_c_cm", "inconsistent"),
    ("Processing after slides ready", "processed_at", "inconsistent"),
    ("Required confirmation missing", "second_pathologist_confirm", "missing"),
    ("Confirmation True with no dysplasia", "second_pathologist_confirm", "inconsistent")
]

results = []
for check, column, error_type in key_rules:
    key_rows = answer_key[
        (answer_key["column"] == column)
        & (answer_key["error_type"] == error_type)
    ]
    documented = df["report_id"].isin(key_rows["report_id"])
    detected = flags[check]
    results.append({
        "Check": check,
        "Found": (documented & detected).sum(),
        "Missed": (documented & ~detected).sum(),
        "Flags without key match": (~documented & detected).sum()
    })

results_table = pd.DataFrame(results).set_index("Check")
print(results_table)

                                     Found  Missed  Flags without key match
Check                                                                      
Missing processing time                  8       0                        0
Missing Prague C                         6       0                        0
Unexpected dysplasia label               2       0                        0
Prague C exceeds M                       3       0                        0
Processing after slides ready            3       0                        0
Required confirmation missing            7       1                        0
Confirmation True with no dysplasia      2       0                        0


## 6. Results and interpretation
There were 158 report records from 80 patients. Patient and report IDs were complete, and report IDs were unique. Eight records (5.06%) lacked processing times, six (3.80%) lacked Prague C measurements, and eight (5.06%) lacked confirmation values.

Six of the seven evaluated checks found every documented issue they targeted. The missing-confirmation rule found seven of eight. No evaluated rule had a flag without a matching answer-key entry for its targeted issue. These statements apply to the selected rules, not all injected errors.

The missed confirmation issue was on a record labeled `LGD`. The confirmation rule recognized `low_grade` but not `LGD`. The record was flagged for its label, but its missing confirmation was not detected by the conditional rule. This shows how a terminology problem can interfere with another check.

Three records had processing times after slide readiness. These records need source review before using that stage's duration. The median receipt-to-report time was 46.42 hours. This is a descriptive result, not evidence of good or poor laboratory performance.

For lab operations, a useful decision is which report to review and which field to verify. The main result to present is the evaluation table: simple rules detected selected documented problems, but one rule missed an issue because it depended on another field's label.

### Follow-up: a blank confirmation with an unexpected grade
The original rule found seven of eight documented missing-confirmation issues. It did not recognize `LGD` as a category requiring confirmation.

I added a separate review check for records where confirmation is blank and the grade is present but outside the expected category list. It means **the confirmation requirement cannot be assessed until the label is reviewed**. It does not establish that confirmation is required.

The original seven checks and their evaluation remain unchanged. The new check uses the report fields, not answer-key values. However, I developed it after reviewing the answer key, so this is a demonstration of a revision on the same dataset, not independent validation.

In [9]:
# Identify uncertainty without changing the original grade or confirmation.
unknown_grade = (
    df["dysplasia_grade"].notna()
    & ~df["dysplasia_grade"].isin(expected_grades)
)
confirmation_needs_review = unknown_grade & confirmation.isna()

followup_review = df.loc[
    confirmation_needs_review,
    ["patient_id", "report_id", "dysplasia_grade",
     "second_pathologist_confirm"]
].copy()
followup_review["Reason for review"] = (
    "Confirmation blank; unexpected grade prevents requirement assessment"
)

print(followup_review.to_string(index=False))
print("Records routed for additional confirmation review:", len(followup_review))

# Add the follow-up to a separate copy; preserve the baseline flags.
revised_flags = flags.copy()
revised_flags["Confirmation requirement needs review"] = confirmation_needs_review
print("Baseline reports with any flag:", flags.any(axis=1).sum())
print("Reports with any flag after follow-up:", revised_flags.any(axis=1).sum())

patient_id    report_id dysplasia_grade second_pathologist_confirm                                                    Reason for review
  SYN-0042 SYN-0042-R03             LGD                        NaN Confirmation blank; unexpected grade prevents requirement assessment
Records routed for additional confirmation review: 1
Baseline reports with any flag: 30
Reports with any flag after follow-up: 30


### What the follow-up shows
The new check routes one record, `SYN-0042-R03`, for additional confirmation review. This report was already flagged for its unexpected `LGD` label, so the unique report count remains 30. The improvement adds a specific reason for investigating its blank confirmation; it does not find a new report.

The baseline result remains seven of eight documented missing-confirmation issues detected by the original conditional rule. The follow-up does not turn that result into eight confirmed detections: an authorized reviewer still needs to verify the category and confirmation requirement.

This illustrates a health-informatics lesson: unexpected terminology should lead to an explicit uncertainty or review state, rather than silently bypassing a dependent check. Original values remain unchanged. Measurement-plausibility issues remain outside the scope of these checks.

## 7. Systems brief: how the checks would be used
The companion `histology_systems_brief.md` gives the short design document required for Track B.

```mermaid
flowchart TD
    A["Synthetic report CSV"] --> B["Documented quality checks"]
    B --> C{"Any flags?"}
    C -->|Yes| D["Authorized staff review source fields"]
    C -->|No| E["Include in summary; quality not guaranteed"]
    D --> F["Document decision and approved correction"]
    F --> G["Run checks again"]
```

A lab technologist or accessioning staff member would review timing and entry questions. A pathologist would review report-content questions. The laboratory data steward would maintain the rules and an audit trail. IT staff would support controlled exports and access.

A flag should prompt verification, not an automatic edit. Keep the original value, the reason for review, the reviewer, and the resolution. In a real LIS, source changes would follow the laboratory's existing approval process.

## 8. Limitations, ethics, and risks
- **Synthetic data:** Error patterns were deliberately created. The findings cannot estimate error rates in a real lab or patient outcomes.
- **Incomplete coverage:** The seven checks do not detect implausible heights, weights, or lengths. Matching IDs cannot prove specimen identity.
- **Dependent rules:** An unexpected label can prevent a conditional check from working. The `LGD` example demonstrates this limitation. The follow-up routes blank confirmation with an unexpected grade for review; it does not determine the requirement automatically.
- **Answer-key leakage:** The initial checks were developed from the README, which describes the injected patterns. This is a demonstration on a documented teaching dataset, not an independent or blinded validation. Future rule changes would need evaluation on a separate dataset.
- **Missingness:** Missing timestamps are not treated as proof that the event happened in order. The turnaround calculation uses only its two endpoints.
- **Workflow differences:** Time includes waiting, staffing, and case complexity. The data cannot separate these causes or justify a performance judgment.
- **Access and privacy:** The supplied files are synthetic and contain no real patient records according to the README. Confirm permission before redistributing instructor-provided files. Real records would require approved access, minimum necessary fields, and secure storage; they would not belong in a public class package.
- **Implementation risk:** Too many flags can create extra work, while missed issues can create false reassurance. Test the rules, assign a reviewer, and monitor unresolved flags before operational use.

I will not claim that passing these checks proves clinical accuracy, correct tissue identity, or that the system is ready for clinical use.

## 9. Conclusion and next steps
Simple pandas checks found selected missing values, inconsistent categories, conflicting measurements, and a workflow-order problem in the synthetic data. The answer key also showed a limitation: an unexpected dysplasia label caused a dependent rule to miss a confirmation issue.

The next step is to review the flagged records with a data steward and document decisions. If the project is extended, define measurement-plausibility rules before testing them on a separate synthetic file. A larger model is not needed for this question.

## 10. References and acknowledgments
1. Instructor-provided `README(4).md`, headed *Histology report data generation*. Source for dataset structure, units, and injected patterns.
2. Instructor-provided `overview.md`, *Term project*, IAN 630. Source for Track B scope and package requirements.
3. Instructor-provided `final.md`, *Final — Presentation and package*, IAN 630. Source for final presentation and submission structure.
4. Instructor-provided report CSV and answer-key CSV, named in Section 2.

The course files also link a quality checklist and AI guidance. Those linked pages were not accessible during this revision. Canvas grading criteria were not supplied.

**AI disclosure:** AI use for python troubleshooting, notebook organization, and proofing explanations. The submitted author is responsible for checking the results and understanding the code. The dataset and injected-error catalog were supplied by the instructor.

## 11. Reproducibility and final submission
Open this notebook from the `histology_project` folder and run the cells from top to bottom. The supplied CSVs must be in its `data` folder with the filenames shown in Section 2. No data generator, internet connection, plotting package, or machine-learning package is needed to run the analysis.

Use Python with pandas and a Jupyter-compatible notebook editor. Install dependencies in your environment before opening the notebook; package installation commands are not part of the analysis.

The cell below records the environment used for this run. Results in this notebook were regenerated from the supplied CSVs.

For Canvas, include this notebook, the short systems brief, and the permitted data files. The course also requires an approximately eight-minute live presentation and two minutes of questions; the notebook does not replace that presentation. Check the Canvas rubric, the final deadline, and the approved midterm scope before submission.

In [10]:
import platform

print("Python:", platform.python_version())
print("pandas:", pd.__version__)

Python: 3.12.14
pandas: 2.2.3
